# CFG 재현 — 진단 셀 보관본

`VQA_RAD_cfg_reproduction.ipynb` 에서 파이프라인이 맞는지 확인하려고 썼던 진단 셀들을 옮겨 둔 보관용 노트북입니다.
지금은 결론이 난 항목들이라 본 노트북에서는 뺐습니다. 다시 쓰려면 본 노트북의 설정 ~ 모델 로드 셀을 먼저 실행한 뒤 이 셀들을 실행하세요.

| 확인 항목 | 결론 |
|---|---|
| LoRA 어댑터 / 추론 루프 (generate vs 수동 루프) | 정상 |
| 이미지 의존성 | 이미지를 실제로 사용함 |
| 학습 수렴 / 학습셋 정확도 | 학습셋 약 99% — 학습은 됐고 일반화가 문제 |
| Med LoRA merge, 체크포인트·비전 타워 가중치, rope/이미지 설정 | 정상 |
| transformers 5.x vs 4.43 logits | 거의 동일 (top1 일치 1.00, 상관 0.9999) |

> Med-only 비교, merge 확인, 기준 실험(5.x ↔ 4.43) 셀은 `DOWNSTREAM_MODE == 'merge_then_new'` 에서만 동작합니다 (`continue` 모드에서는 어댑터를 끄면 Med LoRA 도 꺼지기 때문).


In [ ]:
# ── q_only 가 낮을 때: LoRA 어댑터 / 추론 루프 점검 ─────────────────────────
# 1) 어댑터가 실제로 학습된 가중치인지 (lora_B 는 초기값이 0 → 전부 0 이면 학습 안 된 어댑터)
print('OUTPUT_DIR 파일:', sorted(os.listdir(OUTPUT_DIR))[:15])
_lb = [(n, p) for n, p in model.named_parameters() if 'lora_B' in n]
_tot = sum(p.detach().abs().sum().item() for _, p in _lb)
print(f'lora_B 텐서 {len(_lb)}개, |B| 합 = {_tot:.4f}  (0 이면 학습 안 된 어댑터)')
print('peft_config:', {k: (v.r, v.lora_alpha, sorted(v.target_modules) if not isinstance(v.target_modules, str) else v.target_modules)
                        for k, v in model.peft_config.items()})

# 2) 같은 질문을 generate(batch 1, 베이스라인 방식) 와 수동 루프로 각각 생성해 비교
def gen_single(ann):
    image = Image.open(os.path.join(IMG_ROOT, ann['img_id'])).convert('RGB')
    prompt = processor.tokenizer.apply_chat_template(
        [{'role': 'user', 'content': f"<|image_1|>\n{ann['question']}"}],
        tokenize=False, add_generation_prompt=True)
    inputs = processor(prompt, [image], return_tensors='pt', qs_highlighted_parts=[]).to(DEVICE)
    inputs.pop('highlight_attention_mask', None)
    end_id = processor.tokenizer.convert_tokens_to_ids('<|end|>')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=1, do_sample=False,
                             eos_token_id=[processor.tokenizer.eos_token_id, end_id],
                             use_cache=False)   # 베이스라인 노트북과 동일: 5.x 에서는 KV 캐시 generate 가 깨짐
    return processor.tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip().strip('.')

_n_same = 0
for i in range(10):
    a = _abl[i]
    loop_pred = abl['q_only'][i]['pred']
    try:
        gen_pred = gen_single(a)
    except Exception as e:
        gen_pred = f'<generate 실패: {type(e).__name__}: {str(e)[:80]}>'
    _n_same += int(gen_pred == loop_pred)
    print(f"GT={a['answer']!r:28s} loop={loop_pred!r:28s} generate={gen_pred!r}")
print(f'loop 와 generate 가 같은 예측: {_n_same}/10')


In [ ]:
# ── 이미지를 실제로 쓰는지 점검 ─────────────────────────────────────────────
# 예측이 조건(q_only/cap_only/expert)과 상관없이 거의 같고 closed 가 대부분 'no' 라면
# 모델이 이미지를 못 보고 텍스트 prior 로만 답하고 있을 가능성이 있다.
def img_probe(image, question):
    prompt = processor.tokenizer.apply_chat_template(
        [{'role': 'user', 'content': f'<|image_1|>\n{question}'}],
        tokenize=False, add_generation_prompt=True)
    inputs = processor(prompt, [image], return_tensors='pt', qs_highlighted_parts=[]).to(DEVICE)
    inputs.pop('highlight_attention_mask', None)
    n_img = int((inputs['input_ids'] < 0).sum())
    hh, ww = (inputs['image_sizes'][0] // 336).tolist()
    expected = int((hh * ww + 1) * 144 + 1 + (hh + 1) * 12)
    with torch.no_grad():
        lg = model(**inputs, use_cache=False).logits[0, -1].float()
    return lg, n_img, expected, inputs

_a = _abl[0]
_real = Image.open(os.path.join(IMG_ROOT, _a['img_id'])).convert('RGB')
_black = Image.new('RGB', _real.size, (0, 0, 0))
_other = Image.open(os.path.join(IMG_ROOT, _abl[5]['img_id'])).convert('RGB')

lg_r, n_img, expected, inp = img_probe(_real, _a['question'])
print(f"이미지 토큰 수: 실제={n_img}, 공식 기준 예상={expected}  | 전체 길이={inp['input_ids'].shape[1]}")
pv = inp['pixel_values']
print(f"pixel_values: shape={tuple(pv.shape)}, dtype={pv.dtype}, mean={pv.float().mean():.3f}, std={pv.float().std():.3f}")
print(f"image_sizes={inp['image_sizes'].tolist()}")

lg_b, *_ = img_probe(_black, _a['question'])
lg_o, *_ = img_probe(_other, _a['question'])
tk = processor.tokenizer
print(f"질문: {_a['question']}  | GT: {_a['answer']}")
print(f"  실제 이미지  top1={tk.decode(lg_r.argmax())!r}")
print(f"  검은 이미지  top1={tk.decode(lg_b.argmax())!r}  | 실제와 logits 최대 차이={(lg_r-lg_b).abs().max():.3f}")
print(f"  다른 이미지  top1={tk.decode(lg_o.argmax())!r}  | 실제와 logits 최대 차이={(lg_r-lg_o).abs().max():.3f}")
print('→ 차이가 ~0 이면 이미지가 출력에 반영되지 않는 것')


In [ ]:
# ── 학습이 수렴했는지: trainer_state.json 의 loss 기록 ─────────────────────
_ck = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-*')), key=lambda x: int(x.rsplit('-', 1)[1]))
print('체크포인트:', [os.path.basename(x) for x in _ck])
_st = None
for _d in reversed(_ck):
    _f = os.path.join(_d, 'trainer_state.json')
    if os.path.exists(_f):
        _st = json.load(open(_f)); print('사용:', _f); break
if _st is None:
    print('trainer_state.json 없음')
else:
    _logs = [l for l in _st['log_history'] if 'loss' in l]
    print(f"global_step={_st.get('global_step')}, epoch={_st.get('epoch')}, logging 횟수={len(_logs)}")
    for l in _logs:
        print(f"  step {l['step']:4d}  epoch {l.get('epoch', float('nan')):5.2f}  loss {l['loss']:.4f}  lr {l.get('learning_rate', float('nan')):.2e}")


In [ ]:
# ── 학습에 쓴 문항으로 평가: 파이프라인 불일치인지, 일반화 문제인지 가르는 점검 ──────
# 학습 loss 가 0.05 까지 내려갔으므로, 학습 문항은 거의 맞혀야 한다.
#  - 학습 문항 정확도 높음 + test 낮음 → 추론 파이프라인은 일관됨, 과적합/일반화 문제
#  - 학습 문항 정확도도 낮음          → 학습 때와 추론 때의 입력/처리가 다름 (파이프라인 불일치)
with open(_GDRIVE_JSON, encoding='utf-8') as f:
    _raw = json.load(f)
_TEST_TYPES = {'test_freeform', 'test_para'}
_train_all = [r for r in _raw if str(r.get('phrase_type', '')).lower() not in _TEST_TYPES]
print(f'raw={len(_raw)}, train={len(_train_all)}')   # 베이스라인 노트북: train=1797

random.seed(0)
_tr = random.sample(_train_all, 60)
_recs = []
for r in tqdm(_tr, desc='train-subset'):
    image = Image.open(os.path.join(IMG_ROOT, r['image_name'])).convert('RGB')
    pred, _ = generate_expert_cfg(model, processor, image, str(r['question']).strip(), '', [],
                                  cfg=1.0, max_new_tokens=MAX_NEW_TOKENS, device=DEVICE)
    gt = str(r['answer']).strip()
    at = answer_type_of(gt)
    sc = (closed_accuracy if at == 'CLOSED' else open_recall)(pred, gt)
    _recs.append({'q': r['question'], 'gt': gt, 'pred': pred, 'answer_type': at, 'score': sc})
_m = summarize(_recs)
print(f"학습 문항 q_only  open={_m['open']:.2f}  closed={_m['closed']:.2f}  overall={_m['overall']:.2f}"
      f"  (n_open={_m['n_open']}, n_closed={_m['n_closed']})")
for x in _recs[:8]:
    print(f"GT={x['gt']!r:30s} PRED={x['pred']!r}")


In [ ]:
# ── 다운스트림 LoRA 를 끈 Med-only 와 비교 (같은 test 60문항, 질문만) ───────────────
# 파인튜닝이 test 일반화를 오히려 해치는지 확인: Med-only(제로샷)가 파인튜닝 모델과 비슷하거나 낫다면
# 학습 문항 암기(과적합)가 원인이다.
assert DOWNSTREAM_MODE == 'merge_then_new', '이 진단은 merge_then_new 모드 전용 (continue 모드에서는 어댑터를 끄면 Med LoRA 도 꺼짐)'
def run_q_only(name, anns):
    recs = []
    for ann in tqdm(anns, desc=name, leave=False):
        image = Image.open(os.path.join(IMG_ROOT, ann['img_id'])).convert('RGB')
        pred, _ = generate_expert_cfg(model, processor, image, ann['question'], '', [],
                                      cfg=1.0, max_new_tokens=MAX_NEW_TOKENS, device=DEVICE)
        sc = (closed_accuracy if ann['answer_type'] == 'CLOSED' else open_recall)(pred, ann['answer'])
        recs.append({'q': ann['question'], 'gt': ann['answer'], 'pred': pred,
                     'answer_type': ann['answer_type'], 'score': sc})
    m = summarize(recs)
    print(f"{name:10s} open={m['open']:6.2f}  closed={m['closed']:6.2f}  overall={m['overall']:6.2f}"
          f"  (n_open={m['n_open']}, n_closed={m['n_closed']})")
    return recs

with model.disable_adapter():
    med_only = run_q_only('Med-only', _abl)
for i in range(6):
    print(f"GT={_abl[i]['answer']!r:28s} Med-only={med_only[i]['pred'][:60]!r}  | 파인튜닝={abl['q_only'][i]['pred']!r}")

_mo = summarize(med_only); _ft = summarize(abl['q_only'])
print()
print('===== RESULT (이 블록만 복사해 주세요) =====')
print(f"Med-only  : open={_mo['open']:.2f} closed={_mo['closed']:.2f} overall={_mo['overall']:.2f}")
print(f"파인튜닝  : open={_ft['open']:.2f} closed={_ft['closed']:.2f} overall={_ft['overall']:.2f}")
print('===========================================')


In [ ]:
# ── Med LoRA 가 실제로 base 가중치에 merge 되었는지 확인 ─────────────────────────
# 원본 Phi-3.5-vision 가중치(safetensors)와 현재 모델의 base_layer 가중치를 비교한다.
# 완전히 같으면(max|Δ|=0) Med LoRA 가 적용되지 않은 것.
assert DOWNSTREAM_MODE == 'merge_then_new', '이 진단은 merge_then_new 모드 전용 (continue 모드에서는 어댑터를 끄면 Med LoRA 도 꺼짐)'
from safetensors import safe_open

_wm = json.load(open(os.path.join(BASE_MODEL_DIR, 'model.safetensors.index.json')))['weight_map']
_mods = dict(model.named_modules())

def _cur_weight(suffix):
    name = [n for n in _mods if n.endswith(suffix)][0]
    m = _mods[name]
    return name, (m.base_layer.weight if hasattr(m, 'base_layer') else m.weight).detach().cpu()

for key, suffix in [('model.layers.0.self_attn.qkv_proj.weight', 'layers.0.self_attn.qkv_proj'),
                    ('model.layers.31.mlp.down_proj.weight',    'layers.31.mlp.down_proj'),
                    ('lm_head.weight',                          'lm_head')]:
    with safe_open(os.path.join(BASE_MODEL_DIR, _wm[key]), framework='pt') as f:
        w0 = f.get_tensor(key)
    name, w1 = _cur_weight(suffix)
    d = (w1.float() - w0.float()).abs()
    print(f"{key:45s} max|Δ|={d.max().item():.5f}  mean|Δ|={d.mean().item():.6f}  (원본 mean|w|={w0.float().abs().mean().item():.5f})")


In [ ]:
# ── transformers 5.x 에서 조용히 달라질 수 있는 설정 점검 (rope / 이미지 임베딩) ──────
# 오류 없이 문장은 정상으로 나오지만 품질만 떨어지는 유형의 문제를 찾는다.
_cfg = model.get_base_model().config
print('config.rope_scaling   =', {k: (v[:4] if isinstance(v, list) else v) for k, v in (getattr(_cfg, 'rope_scaling', None) or {}).items()} or None)
_rp = getattr(_cfg, 'rope_parameters', None)
print('config.rope_parameters=', {k: (v[:4] if isinstance(v, list) else v) for k, v in _rp.items()} if isinstance(_rp, dict) else _rp)
print('original_max_position_embeddings =', getattr(_cfg, 'original_max_position_embeddings', None),
      '| max_position_embeddings =', getattr(_cfg, 'max_position_embeddings', None))

_mods = dict(model.named_modules())
_rot = [n for n in _mods if n.endswith('layers.0.self_attn.rotary_emb')]
if _rot:
    r = _mods[_rot[0]]
    print('rotary 클래스:', type(r).__name__)
    for a in ('short_factor', 'long_factor', 'original_max_position_embeddings', 'max_position_embeddings', 'base', 'scaling_factor'):
        v = getattr(r, a, None)
        if v is not None:
            print(f"  {a} =", (list(v)[:4] if hasattr(v, '__len__') and not isinstance(v, str) else v))
    if hasattr(r, 'inv_freq'):
        print('  inv_freq[:4] =', r.inv_freq[:4].tolist(), '| dtype', r.inv_freq.dtype)
else:
    print('layers.0.self_attn.rotary_emb 모듈을 찾지 못함')

_ve = [n for n in _mods if n.endswith('vision_embed_tokens')]
if _ve:
    v = _mods[_ve[0]]
    print('vision_embed_tokens 클래스:', type(v).__name__)
    for a in ('use_hd_transform', 'with_learnable_separator', 'hd_transform_order', 'num_img_tokens',
              'type_feature', 'img_processor_padding', 'image_dim_out'):
        print(f"  {a} =", getattr(v, a, '<없음>'))
    for a in ('glb_GN', 'sub_GN'):
        t = getattr(v, a, None)
        print(f"  {a} =", None if t is None else (tuple(t.shape), f"mean={t.float().mean().item():.4f}"))
print('embd_layer(config) =', getattr(_cfg, 'embd_layer', '<없음>'))


In [ ]:
# ── 체크포인트 가중치가 모델에 그대로 로드되었는지 (비전 타워 / 임베딩 / norm) ─────────────
# LoRA 가 닿지 않는 모듈은 원본 safetensors 와 비트 단위로 같아야 한다.
# 다르거나 모델에 없으면 5.x 로딩 과정에서 가중치가 빠졌거나 랜덤 초기화된 것.
from safetensors import safe_open

_wm = json.load(open(os.path.join(BASE_MODEL_DIR, 'model.safetensors.index.json')))['weight_map']
_cur = {n.replace('base_model.model.', '', 1): p for n, p in model.named_parameters()}
_cur.update({n.replace('base_model.model.', '', 1): b for n, b in model.named_buffers()})

def _want(k):
    return (k.startswith('model.vision_embed_tokens') or k in ('model.embed_tokens.weight', 'model.norm.weight')
            or 'layernorm' in k)

_stat = {'equal': 0, 'diff': 0, 'missing': 0}
_bad = []
_by_part = {}
for fn in sorted(set(_wm.values())):
    with safe_open(os.path.join(BASE_MODEL_DIR, fn), framework='pt') as f:
        for k in f.keys():
            if not _want(k):
                continue
            part = 'vision' if 'vision_embed_tokens' in k else 'text'
            _by_part.setdefault(part, [0, 0])
            if k not in _cur:
                _stat['missing'] += 1; _bad.append(('missing', k)); _by_part[part][1] += 1
                continue
            w0, w1 = f.get_tensor(k), _cur[k].detach().cpu()
            if w0.shape == w1.shape and torch.equal(w0.to(w1.dtype), w1):
                _stat['equal'] += 1; _by_part[part][0] += 1
            else:
                _stat['diff'] += 1; _by_part[part][1] += 1
                _bad.append(('diff', k, tuple(w0.shape), tuple(w1.shape),
                             float((w0.float() - w1.float()).abs().max()) if w0.shape == w1.shape else None))
print('결과:', _stat)
print('부분별 (같음, 다르거나 없음):', _by_part)
for b in _bad[:12]:
    print('  ', b)


In [ ]:
# ── 비전 타워(CLIP) 가중치: 이름이 달라서 못 찾은 것인지, 실제로 로드가 안 된 것인지 ──────────
# 앞 셀에서 img_processor.vision_model.* 391개가 "모델에 없음"으로 나왔다.
# (a) 모델 안의 이름만 다르고 값은 로드됨  → 문제 아님
# (b) 값이 체크포인트와 다름(랜덤 초기화)   → 이미지 특징이 망가진 것
_vn = [n for n in _cur if 'vision_embed_tokens' in n]
print('모델 내 vision_embed_tokens 파라미터/버퍼 수:', len(_vn))
print('img_processor 포함 수:', sum('img_processor' in n for n in _vn))
for n in _vn[:6]:
    print('  ', n, tuple(_cur[n].shape))
_pe = [n for n in _vn if 'patch_embedding' in n]
print('patch_embedding 이름:', _pe)

_res = {'equal': 0, 'diff': 0, 'no_candidate': 0, 'ambiguous': 0}
_diffs = []
for fn in sorted(set(_wm.values())):
    with safe_open(os.path.join(BASE_MODEL_DIR, fn), framework='pt') as f:
        for k in f.keys():
            if 'img_processor' not in k:
                continue
            suffix = k.split('img_processor.', 1)[1]
            if suffix.startswith('vision_model.'):      # 체크포인트: img_processor.vision_model.*  /  모델: img_processor.*
                suffix = suffix[len('vision_model.'):]
            cands = [n for n in _vn if n.endswith(suffix)]
            if not cands:
                _res['no_candidate'] += 1; continue
            if len(cands) > 1:
                _res['ambiguous'] += 1; continue
            w0, w1 = f.get_tensor(k), _cur[cands[0]].detach().cpu()
            if w0.shape == w1.shape and torch.equal(w0.to(w1.dtype), w1):
                _res['equal'] += 1
            else:
                _res['diff'] += 1
                _diffs.append((k, cands[0], float((w0.float() - w1.float()).abs().max()) if w0.shape == w1.shape else 'shape'))
print('suffix 매칭 비교:', _res)
for d in _diffs[:8]:
    print('  diff:', d)

print()
print('===== RESULT (이 블록만 복사해 주세요) =====')
print('suffix 매칭 비교:', _res)
print('diff 예시:', _diffs[:3])
print('===========================================')


In [ ]:
# ── [기준 실험 1/3] 현재 환경(transformers 5.x)에서 Med-only 첫 토큰 logits 저장 ──────────
# 다운스트림 LoRA 를 끈 상태(= Med LoRA 만 merge 된 모델)로 test 앞 N_REF 문항을 질문만으로 돌린다.
assert DOWNSTREAM_MODE == 'merge_then_new', '이 진단은 merge_then_new 모드 전용 (continue 모드에서는 어댑터를 끄면 Med LoRA 도 꺼짐)'
N_REF = 20
LOGITS_5X  = '/content/drive/MyDrive/expert_cfg/logits_5x.npz'
LOGITS_443 = '/content/drive/MyDrive/expert_cfg/logits_443.npz'

_lg, _st = [], []
with model.disable_adapter():
    for ann in tqdm(test_anns[:N_REF], desc='5.x logits'):
        image = Image.open(os.path.join(IMG_ROOT, ann['img_id'])).convert('RGB')
        prompt = processor.tokenizer.apply_chat_template(
            [{'role': 'user', 'content': '<|image_1|>\n' + ann['question']}],
            tokenize=False, add_generation_prompt=True)
        inputs = processor(prompt, [image], return_tensors='pt', qs_highlighted_parts=[]).to(DEVICE)
        inputs.pop('highlight_attention_mask', None)
        with torch.no_grad():
            lg = model(**inputs, use_cache=False).logits[0, -1].float().cpu().numpy()
        pv = inputs['pixel_values'].float()
        _st.append([pv.mean().item(), pv.std().item(), float(inputs['input_ids'].shape[1]), float(pv[0, 0].sum().item())])
        _lg.append(lg)
import numpy as np
np.savez(LOGITS_5X, logits=np.stack(_lg), stats=np.array(_st))
print('저장:', LOGITS_5X)


In [ ]:
# ── [기준 실험 2/3] Python 3.10 + transformers 4.43 환경에서 같은 logits 저장 ───────────────
# repo 가 전제하는 환경(requirements.txt)에 가깝게 별도 venv 를 만들어 같은 입력을 돌린다.
# 시간이 걸린다 (torch 2.3.0 설치 포함). 이미 만들어져 있으면 건너뛴다.
import subprocess, sys

DUMP_SCRIPT = r"""
import argparse, json, os, sys
import numpy as np
import torch
from PIL import Image
ap = argparse.ArgumentParser()
ap.add_argument('--repo'); ap.add_argument('--base'); ap.add_argument('--med_lora')
ap.add_argument('--test_json'); ap.add_argument('--img_root')
ap.add_argument('--n', type=int, default=20); ap.add_argument('--out')
a = ap.parse_args()
os.chdir(a.repo); sys.path.insert(0, a.repo)
import transformers, peft
print('transformers', transformers.__version__, '| peft', peft.__version__, '| torch', torch.__version__)
from transformers import AutoModelForCausalLM, AutoProcessor
from peft import PeftModel
processor = AutoProcessor.from_pretrained(a.base, trust_remote_code=True, num_crops=16)
model = AutoModelForCausalLM.from_pretrained(a.base, torch_dtype=torch.bfloat16, trust_remote_code=True,
                                             _attn_implementation='eager')
model = PeftModel.from_pretrained(model, a.med_lora).merge_and_unload().to('cuda').eval()
anns = json.load(open(a.test_json, encoding='utf-8'))[:a.n]
logits_all, stats = [], []
for ann in anns:
    image = Image.open(os.path.join(a.img_root, ann['img_id'])).convert('RGB')
    prompt = processor.tokenizer.apply_chat_template(
        [{'role': 'user', 'content': '<|image_1|>\n' + ann['question']}],
        tokenize=False, add_generation_prompt=True)
    inputs = processor(prompt, [image], return_tensors='pt', qs_highlighted_parts=[]).to('cuda')
    inputs.pop('highlight_attention_mask', None)
    with torch.no_grad():
        lg = model(**inputs, use_cache=False).logits[0, -1].float().cpu().numpy()
    pv = inputs['pixel_values'].float()
    stats.append([pv.mean().item(), pv.std().item(), float(inputs['input_ids'].shape[1]), float(pv[0, 0].sum().item())])
    logits_all.append(lg)
np.savez(a.out, logits=np.stack(logits_all), stats=np.array(stats))
print('saved', a.out)
"""
SCRIPT_PATH = '/content/dump_logits.py'
with open(SCRIPT_PATH, 'w') as f:
    f.write(DUMP_SCRIPT)

VENV_PY = '/content/venv443/bin/python'
if not os.path.exists(VENV_PY):
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'uv'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '/content/venv443', '--python', '3.10'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', VENV_PY, '-q',
                    'torch==2.3.0', 'torchvision==0.18.0', 'transformers==4.43.0', 'accelerate==0.30.0',
                    'peft==0.11.1', 'numpy==1.26.4', 'Pillow==10.3.0', 'safetensors', 'sentencepiece', 'protobuf'],
                   check=True)

# uv venv 에는 setuptools 가 없어 peft(torch.utils.cpp_extension) import 가 실패한다 → 이미 만든 venv 에도 설치
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', VENV_PY, '-q', 'setuptools', 'wheel'], check=True)

_env = dict(os.environ, HF_MODULES_CACHE='/content/hf_modules_443', PYTHONNOUSERSITE='1')
# 같은 GPU 를 쓰므로 이 커널의 모델을 잠시 CPU 로 내리고 캐시를 비운다 (안 그러면 하위 프로세스가 OOM)
import gc
print(f'실행 전 GPU 할당/예약: {torch.cuda.memory_allocated()/2**30:.1f} / {torch.cuda.memory_reserved()/2**30:.1f} GiB')
model.to('cpu')
gc.collect(); torch.cuda.empty_cache()
print(f'모델 CPU 이동 후 GPU 예약: {torch.cuda.memory_reserved()/2**30:.1f} GiB')
try:
    _r = subprocess.run([VENV_PY, SCRIPT_PATH, '--repo', str(REPO_ROOT), '--base', os.path.abspath(BASE_MODEL_DIR),
                         '--med_lora', os.path.abspath(med_lora_path), '--test_json', TEST_JSON,
                         '--img_root', IMG_ROOT, '--n', str(N_REF), '--out', LOGITS_443],
                        env=_env, capture_output=True, text=True)
finally:
    model.to(DEVICE)
    print('커널 모델을 GPU 로 복귀')
print(_r.stdout[-2500:])
print(_r.stderr[-2500:])

print()
print('===== RESULT (이 블록만 복사해 주세요) =====')
print('returncode =', _r.returncode)
print('stdout 마지막 줄들:')
print(*_r.stdout.strip().splitlines()[-4:], sep=chr(10))
print('stderr 마지막 줄들:')
print(*_r.stderr.strip().splitlines()[-6:], sep=chr(10))
print('===========================================')


In [ ]:
# ── [기준 실험 3/3] 두 환경의 logits 비교 ─────────────────────────────────────────────
import numpy as np
_a, _b = np.load(LOGITS_5X), np.load(LOGITS_443)
l5, l4 = _a['logits'], _b['logits']
d = np.abs(l5 - l4)
top1_same = float((l5.argmax(1) == l4.argmax(1)).mean())
corr = float(np.mean([np.corrcoef(x, y)[0, 1] for x, y in zip(l5, l4)]))
tk = processor.tokenizer
print('샘플별 (5.x top1 | 4.43 top1 | 최대 |Δlogit|):')
for i in range(len(l5)):
    print(f"  {i:2d}  {tk.decode(int(l5[i].argmax()))!r:14s} {tk.decode(int(l4[i].argmax()))!r:14s} {d[i].max():7.3f}")
print()
print('===== RESULT (이 블록만 복사해 주세요) =====')
print(f'top1 일치율={top1_same:.2f} | 최대|Δlogit| 평균={d.max(1).mean():.3f}, 최대={d.max():.3f} | 전체 상관={corr:.4f}')
print('입력 통계 [pv.mean, pv.std, 토큰길이, 첫 crop 합]')
print('  5.x :', np.round(_a['stats'][:3], 4).tolist())
print('  4.43:', np.round(_b['stats'][:3], 4).tolist())
print('===========================================')
